In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from statsmodels.tsa.filters.hp_filter import hpfilter
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, SimpleRNN, Dense, Conv1D, Flatten, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from itertools import combinations
import warnings
warnings.filterwarnings("ignore")

# ================================================
# ENHANCED INFLATION FORECASTING - MAINTAINING ORIGINAL ARCHITECTURE
# Addressing Reviewer Comments while preserving original structure
# ================================================

print("🚀 ENHANCED INFLATION FORECASTING ANALYSIS")
print("="*60)
print("✅ Original architecture preserved")
print("✅ Multiple evaluation metrics added (MSE, MAE, R², MAPE)")  
print("✅ Model configuration details documented")
print("✅ ML models comparison included")
print("✅ Enhanced preprocessing documentation")
print("="*60)

# ================================================
# 1. DETAILED MODEL CONFIGURATIONS (Addressing Reviewer Comment #6)
# ================================================
MODEL_CONFIGS = {
    'LSTM': {
        'units': 50,
        'activation': 'tanh',
        'dropout_rate': 0.2,
        'epochs': 100,
        'batch_size': 32,
        'layers': 2,
        'architecture': 'Sequential with 2 LSTM layers + Dense output'
    },
    'GRU': {
        'units': 50,
        'activation': 'tanh', 
        'dropout_rate': 0.2,
        'epochs': 100,
        'batch_size': 32,
        'layers': 2,
        'architecture': 'Sequential with 2 GRU layers + Dense output'
    },
    'CNN': {
        'filters': 64,
        'kernel_size': 2,
        'activation': 'relu',
        'dense_units': 50,
        'dropout_rate': 0.2,
        'epochs': 100,
        'batch_size': 32,
        'input_type': '1D convolution',
        'architecture': 'Conv1D + Flatten + Dense layers'
    },
    'RNN': {
        'units': 50,
        'activation': 'tanh',
        'dropout_rate': 0.2,
        'epochs': 100,
        'batch_size': 32,
        'layers': 2,
        'architecture': 'Sequential with 2 SimpleRNN layers + Dense output'
    },
    'ANN': {
        'units': 50,
        'activation': 'relu',
        'dropout_rate': 0.2,
        'epochs': 100,
        'batch_size': 32,
        'layers': 2,
        'architecture': 'Fully connected Dense layers'
    }
}

print("\n📋 MODEL ARCHITECTURE DETAILS:")
for model_name, config in MODEL_CONFIGS.items():
    print(f"   {model_name}: {config['architecture']}")
    print(f"      - Units/Filters: {config.get('units', config.get('filters'))}")
    print(f"      - Activation: {config['activation']}")
    print(f"      - Dropout: {config['dropout_rate']}")
    print(f"      - Epochs: {config['epochs']}")

# ================================================
# 2. DATA PREPROCESSING WITH DOCUMENTATION (Addressing Reviewer Comment #5)
# ================================================
def load_and_document_data(file_path):
    """
    Load and document inflation data preprocessing steps
    
    Preprocessing steps applied:
    1. Load CSV data
    2. Handle missing values via interpolation
    3. Apply HP filter for trend extraction
    4. Normalize using MinMaxScaler or StandardScaler
    
    Parameters:
    -----------
    file_path : str
        Path to the inflation parameters CSV file
        
    Returns:
    --------
    pd.DataFrame : Loaded and initial processed data
    """
    print("\n📊 DATA LOADING AND PREPROCESSING DOCUMENTATION")
    print("-" * 50)
    
    # Load data
    data = pd.read_csv(file_path)
    print(f"✅ Data loaded: {data.shape[0]} observations, {data.shape[1]} features")
    
    # Missing values handling
    missing_before = data.isnull().sum().sum()
    print(f"📋 Missing values before: {missing_before}")
    
    if missing_before > 0:
        # Apply linear interpolation for missing values
        data = data.interpolate(method='linear')
        data = data.fillna(method='ffill').fillna(method='bfill')
        
    missing_after = data.isnull().sum().sum()
    print(f"✅ Missing values after interpolation: {missing_after}")
    print(f"📊 Final dataset shape: {data.shape}")
    
    return data

# Load the data with documentation
merged_data = load_and_document_data("C:\\Users\\elect\\OneDrive\\Documentos\\Doctorado\\Articulo 2\\Data\\Inflation_Parameters.csv")

# Function to apply HP Filter with error handling
def apply_hp_filter(data, lamb=129600):
    """
    Apply Hodrick-Prescott filter for trend extraction
    
    Parameters:
    -----------
    data : array-like
        Time series data to filter
    lamb : float  
        HP filter smoothing parameter (129600 for monthly data)
        
    Returns:
    --------
    np.array : Trend component from HP decomposition
    """
    try:
        cycle, trend = hpfilter(data, lamb=lamb)
        return trend
    except Exception as e:
        print(f"⚠️ HP Filter warning for series: {e}")
        return data  # Return original if filtering fails

# Function to create sequences (MAINTAINED ORIGINAL)
def create_sequences(data, seq_length):
    """
    Create sequences for time series prediction
    
    Input structure:
    - Each sequence contains seq_length time steps
    - Features: all columns except target (last column)
    - Target: last column at time t+seq_length
    
    Parameters:
    -----------
    data : np.array
        Scaled time series data [samples, features]
    seq_length : int
        Length of input sequences (lookback window)
        
    Returns:
    --------
    tuple : (X, y) where X is [samples, seq_length, features] and y is [samples]
    """
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i + seq_length, :-1])  # All features except the target
        y.append(data[i + seq_length - 1, -1])  # The target value
    return np.array(X), np.array(y)

# ================================================
# 3. MODEL BUILDERS (MAINTAINING ORIGINAL ARCHITECTURE)
# ================================================
def build_lstm_model(input_shape):
    """
    Build LSTM model with original architecture + enhancements
    
    Architecture:
    - LSTM Layer 1: 50 units, return_sequences=True, activation=tanh
    - LSTM Layer 2: 50 units, activation=tanh  
    - Dense Output: 1 unit (regression)
    - Enhancements: Dropout for regularization
    """
    model = Sequential()
    model.add(LSTM(50, return_sequences=True, input_shape=input_shape, activation='tanh'))
    model.add(Dropout(0.2))  # Added for overfitting control
    model.add(LSTM(50, activation='tanh'))
    model.add(Dropout(0.2))  # Added for overfitting control
    model.add(Dense(1))
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

def build_gru_model(input_shape):
    """
    Build GRU model with original architecture + enhancements
    
    Architecture:
    - GRU Layer 1: 50 units, return_sequences=True, activation=tanh
    - GRU Layer 2: 50 units, activation=tanh
    - Dense Output: 1 unit (regression)
    - Enhancements: Dropout for regularization
    """
    model = Sequential()
    model.add(GRU(50, return_sequences=True, input_shape=input_shape, activation='tanh'))
    model.add(Dropout(0.2))  # Added for overfitting control
    model.add(GRU(50, activation='tanh'))
    model.add(Dropout(0.2))  # Added for overfitting control
    model.add(Dense(1))
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

def build_cnn_model(input_shape):
    """
    Build CNN model with original architecture + enhancements
    
    Architecture:
    - Conv1D: 64 filters, kernel_size=2, activation=ReLU (1D convolution)
    - Flatten: Convert to 1D
    - Dense: 50 units, activation=ReLU
    - Dense Output: 1 unit (regression)
    - Enhancements: Dropout for regularization
    """
    model = Sequential()
    model.add(Conv1D(64, kernel_size=2, activation='relu', input_shape=input_shape))
    model.add(Dropout(0.2))  # Added for overfitting control
    model.add(Flatten())
    model.add(Dense(50, activation='relu'))
    model.add(Dropout(0.2))  # Added for overfitting control
    model.add(Dense(1))
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

def build_rnn_model(input_shape):
    """
    Build RNN model with original architecture + enhancements
    
    Architecture:
    - SimpleRNN Layer 1: 50 units, return_sequences=True, activation=tanh
    - SimpleRNN Layer 2: 50 units, activation=tanh
    - Dense Output: 1 unit (regression)
    - Enhancements: Dropout for regularization
    """
    model = Sequential()
    model.add(SimpleRNN(50, return_sequences=True, input_shape=input_shape, activation='tanh'))
    model.add(Dropout(0.2))  # Added for overfitting control
    model.add(SimpleRNN(50, activation='tanh'))
    model.add(Dropout(0.2))  # Added for overfitting control
    model.add(Dense(1))
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

def build_ann_model(input_shape):
    """
    Build ANN model with original architecture + enhancements
    
    Architecture:
    - Dense Layer 1: 50 units, activation=ReLU
    - Dense Layer 2: 50 units, activation=ReLU  
    - Dense Output: 1 unit (regression)
    - Enhancements: Dropout for regularization
    """
    model = Sequential()
    model.add(Dense(50, activation='relu', input_shape=input_shape))
    model.add(Dropout(0.2))  # Added for overfitting control
    model.add(Dense(50, activation='relu'))
    model.add(Dropout(0.2))  # Added for overfitting control
    model.add(Dense(1))
    model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

# ================================================
# 4. COMPREHENSIVE METRICS FUNCTION (Addressing Reviewer Comment #7)
# ================================================
def calculate_comprehensive_metrics(y_true, y_pred):
    """
    Calculate multiple evaluation metrics as requested by reviewers
    
    Metrics included:
    - MSE: Mean Squared Error
    - RMSE: Root Mean Squared Error  
    - MAE: Mean Absolute Error
    - R²: R-squared (coefficient of determination)
    - MAPE: Mean Absolute Percentage Error
    
    Parameters:
    -----------
    y_true : array-like
        True target values
    y_pred : array-like  
        Predicted target values
        
    Returns:
    --------
    dict : Dictionary containing all metrics
    """
    return {
        'mse': mean_squared_error(y_true, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'mae': mean_absolute_error(y_true, y_pred),
        'r2': r2_score(y_true, y_pred),
        'mape': mean_absolute_percentage_error(y_true, y_pred) * 100
    }

# ================================================
# 5. ML MODELS FOR COMPARISON (Addressing Reviewer Comment #8)
# ================================================
def train_ml_baselines(X_train_flat, X_test_flat, y_train, y_test):
    """
    Train machine learning baseline models for comparison
    
    Models included:
    - Random Forest: Ensemble of decision trees
    - XGBoost: Gradient boosting framework
    
    Returns:
    --------
    dict : Results from baseline ML models
    """
    print("   🌳 Training ML baselines...")
    ml_results = {}
    
    try:
        # Random Forest
        rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
        rf_model.fit(X_train_flat, y_train)
        rf_pred = rf_model.predict(X_test_flat)
        ml_results['random_forest'] = calculate_comprehensive_metrics(y_test, rf_pred)
        
        # XGBoost
        xgb_model = xgb.XGBRegressor(n_estimators=100, random_state=42, n_jobs=-1)
        xgb_model.fit(X_train_flat, y_train)
        xgb_pred = xgb_model.predict(X_test_flat)
        ml_results['xgboost'] = calculate_comprehensive_metrics(y_test, xgb_pred)
        
        print(f"      ✅ Random Forest: MSE={ml_results['random_forest']['mse']:.6f}")
        print(f"      ✅ XGBoost: MSE={ml_results['xgboost']['mse']:.6f}")
        
    except Exception as e:
        print(f"      ⚠️ ML baseline error: {e}")
        
    return ml_results

# ================================================
# 6. MAIN ANALYSIS (MAINTAINING ORIGINAL STRUCTURE)
# ================================================

# Define features and parameters (ORIGINAL)
features = ['USM2', 'USPPIYY', 'INDPRO', 'UNRATE', 'USOIL', 'Annual Inflation Rate']
results = []
seq_length = 10  # Example sequence length (ORIGINAL)

# Test both normalization methods (ENHANCEMENT)
scalers = {
    'MinMax': MinMaxScaler(),
    'Standard': StandardScaler()
}

print(f"\n🔬 STARTING FEATURE COMBINATION ANALYSIS")
print(f"Features: {features[:-1]}")  # Exclude target from display
print(f"Target: {features[-1]}")
print(f"Sequence length: {seq_length}")
print(f"Normalization methods: {list(scalers.keys())}")
print("-" * 60)

# Calculate total combinations for progress tracking
total_combinations = sum(1 for r in range(1, len(features)) for _ in combinations(features[:-1], r))
total_experiments = total_combinations * len(scalers)
current_experiment = 0

# ORIGINAL LOOP STRUCTURE MAINTAINED
for r in range(1, len(features)):  # Exclude target from combinations
    for combo in combinations(features[:-1], r):  # Only use feature columns
        for scaler_name, scaler in scalers.items():  # Test both scalers
            current_experiment += 1
            print(f"\n📊 Experiment {current_experiment}/{total_experiments}: {combo} | {scaler_name}")
            
            try:
                # Prepare feature set (ORIGINAL LOGIC)
                selected_features = list(combo) + ['Annual Inflation Rate']
                selected_data = merged_data[selected_features].copy()

                # Apply HP Filter to each column (ORIGINAL)
                print("   🔧 Applying HP Filter...")
                for column in selected_features:
                    filtered_data = apply_hp_filter(selected_data[column].values.flatten())
                    # Ensure the filtered data matches the original length (ORIGINAL)
                    if len(filtered_data) != len(selected_data[column]):
                        filtered_data = np.resize(filtered_data, len(selected_data[column]))
                    selected_data[column] = filtered_data

                # Normalize and create sequences (ORIGINAL STRUCTURE)
                scaled_data = scaler.fit_transform(selected_data)
                X, y = create_sequences(scaled_data, seq_length)
                
                if len(X) < 10:  # Minimum data check
                    print("   ⚠️ Insufficient data for training")
                    continue
                    
                # Train-test split (ORIGINAL)
                X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
                print(f"   📊 Data split: Train={len(X_train)}, Test={len(X_test)}")

                # Build and evaluate models (ORIGINAL STRUCTURE WITH ENHANCEMENTS)
                input_shape = (seq_length, X_train.shape[2])
                
                # Prepare data for different model types
                X_train_ann = X_train.reshape(X_train.shape[0], -1)  # ANN needs flattened input (ORIGINAL)
                X_test_ann = X_test.reshape(X_test.shape[0], -1)
                
                # Early stopping callback (ENHANCEMENT)
                early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

                # Train models (MAINTAINING ORIGINAL ORDER)
                models_data = []
                
                # LSTM MODEL (ORIGINAL + ENHANCEMENTS)
                print("   🤖 Training LSTM...")
                lstm_model = build_lstm_model(input_shape)
                lstm_history = lstm_model.fit(X_train, y_train, epochs=100, batch_size=32, 
                                            validation_split=0.2, verbose=0, callbacks=[early_stopping])
                lstm_pred = lstm_model.predict(X_test, verbose=0).flatten()
                lstm_metrics = calculate_comprehensive_metrics(y_test, lstm_pred)
                models_data.append(('LSTM', lstm_metrics))

                # GRU MODEL (ORIGINAL + ENHANCEMENTS)
                print("   🤖 Training GRU...")
                gru_model = build_gru_model(input_shape)
                gru_history = gru_model.fit(X_train, y_train, epochs=100, batch_size=32, 
                                          validation_split=0.2, verbose=0, callbacks=[early_stopping])
                gru_pred = gru_model.predict(X_test, verbose=0).flatten()
                gru_metrics = calculate_comprehensive_metrics(y_test, gru_pred)
                models_data.append(('GRU', gru_metrics))

                # CNN MODEL (ORIGINAL + ENHANCEMENTS)  
                print("   🤖 Training CNN...")
                cnn_model = build_cnn_model(input_shape)
                cnn_history = cnn_model.fit(X_train, y_train, epochs=100, batch_size=32, 
                                          validation_split=0.2, verbose=0, callbacks=[early_stopping])
                cnn_pred = cnn_model.predict(X_test, verbose=0).flatten()
                cnn_metrics = calculate_comprehensive_metrics(y_test, cnn_pred)
                models_data.append(('CNN', cnn_metrics))

                # RNN MODEL (ORIGINAL + ENHANCEMENTS)
                print("   🤖 Training RNN...")
                rnn_model = build_rnn_model(input_shape)
                rnn_history = rnn_model.fit(X_train, y_train, epochs=100, batch_size=32, 
                                          validation_split=0.2, verbose=0, callbacks=[early_stopping])
                rnn_pred = rnn_model.predict(X_test, verbose=0).flatten()
                rnn_metrics = calculate_comprehensive_metrics(y_test, rnn_pred)
                models_data.append(('RNN', rnn_metrics))

                # ANN MODEL (ORIGINAL + ENHANCEMENTS)
                print("   🤖 Training ANN...")
                ann_model = build_ann_model((X_train_ann.shape[1],))
                ann_history = ann_model.fit(X_train_ann, y_train, epochs=100, batch_size=32, 
                                          validation_split=0.2, verbose=0, callbacks=[early_stopping])
                ann_pred = ann_model.predict(X_test_ann, verbose=0).flatten()
                ann_metrics = calculate_comprehensive_metrics(y_test, ann_pred)
                models_data.append(('ANN', ann_metrics))
                
                # Train ML baselines (NEW ADDITION)
                X_train_flat = X_train.reshape(X_train.shape[0], -1)
                X_test_flat = X_test.reshape(X_test.shape[0], -1)
                ml_results = train_ml_baselines(X_train_flat, X_test_flat, y_train, y_test)
                
                # Add ML results to models_data
                for ml_model, ml_metrics in ml_results.items():
                    models_data.append((ml_model.upper(), ml_metrics))

                # Store results (ENHANCED FORMAT)
                for model_name, metrics in models_data:
                    result = {
                        'features': combo,
                        'n_features': len(combo),
                        'scaler': scaler_name,
                        'model': model_name,
                        'seq_length': seq_length if model_name not in ['RANDOM_FOREST', 'XGBOOST'] else 'N/A',
                        **metrics  # Unpack all metrics (mse, rmse, mae, r2, mape)
                    }
                    results.append(result)
                    
                    print(f"      ✅ {model_name}: MSE={metrics['mse']:.6f}, R²={metrics['r2']:.4f}")

            except Exception as e:
                print(f"   ❌ Error in experiment: {e}")
                continue

# ================================================
# 7. ENHANCED RESULTS ANALYSIS AND VISUALIZATION
# ================================================
print(f"\n📊 CREATING ENHANCED RESULTS ANALYSIS")
print("-" * 50)

# Convert results to DataFrame
results_df = pd.DataFrame(results)

if len(results_df) > 0:
    print(f"✅ Analysis completed: {len(results_df)} experiments")
    
    # Save enhanced results
    output_path = r'C:\Users\elect\OneDrive\Documentos\Doctorado\Articulo 2\Data\enhanced_model_results_hp.csv'
    results_df.to_csv(output_path, index=False)
    print(f"💾 Results saved to: {output_path}")
    
    # Create enhanced visualization (ADDRESSING REVIEWER COMMENT #4)
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('Enhanced Inflation Forecasting Results\n(Multiple Metrics & Model Comparison)', 
                 fontsize=16, fontweight='bold')
    
    # 1. MSE Comparison by Model
    ax1 = axes[0, 0]
    model_mse = results_df.groupby('model')['mse'].mean().sort_values()
    bars1 = ax1.bar(range(len(model_mse)), model_mse.values, 
                    color=plt.cm.Set3(np.linspace(0, 1, len(model_mse))))
    ax1.set_xticks(range(len(model_mse)))
    ax1.set_xticklabels(model_mse.index, rotation=45, ha='right')
    ax1.set_ylabel('Mean Squared Error')
    ax1.set_title('MSE by Model Type')
    ax1.grid(True, alpha=0.3)
    
    # 2. R² Comparison by Model  
    ax2 = axes[0, 1]
    model_r2 = results_df.groupby('model')['r2'].mean().sort_values(ascending=False)
    bars2 = ax2.bar(range(len(model_r2)), model_r2.values,
                    color=plt.cm.Set2(np.linspace(0, 1, len(model_r2))))
    ax2.set_xticks(range(len(model_r2)))
    ax2.set_xticklabels(model_r2.index, rotation=45, ha='right')
    ax2.set_ylabel('R² Score')
    ax2.set_title('R² by Model Type')
    ax2.grid(True, alpha=0.3)
    
    # 3. MAE Comparison by Model
    ax3 = axes[0, 2]  
    model_mae = results_df.groupby('model')['mae'].mean().sort_values()
    bars3 = ax3.bar(range(len(model_mae)), model_mae.values,
                    color=plt.cm.Pastel1(np.linspace(0, 1, len(model_mae))))
    ax3.set_xticks(range(len(model_mae)))
    ax3.set_xticklabels(model_mae.index, rotation=45, ha='right')
    ax3.set_ylabel('Mean Absolute Error')
    ax3.set_title('MAE by Model Type')
    ax3.grid(True, alpha=0.3)
    
    # 4. Scaler Comparison
    ax4 = axes[1, 0]
    scaler_perf = results_df.groupby(['model', 'scaler'])['r2'].mean().unstack()
    scaler_perf.plot(kind='bar', ax=ax4, width=0.8)
    ax4.set_xlabel('Model Type')
    ax4.set_ylabel('R² Score')
    ax4.set_title('Normalization Method Comparison')
    ax4.legend(title='Scaler')
    ax4.grid(True, alpha=0.3)
    plt.setp(ax4.xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    # 5. Feature Count Impact
    ax5 = axes[1, 1]
    feature_impact = results_df.groupby('n_features')['r2'].agg(['mean', 'std']).reset_index()
    ax5.errorbar(feature_impact['n_features'], feature_impact['mean'], 
                yerr=feature_impact['std'], marker='o', capsize=5, linewidth=2)
    ax5.set_xlabel('Number of Features')
    ax5.set_ylabel('R² Score (Mean ± Std)')
    ax5.set_title('Performance vs Feature Count')
    ax5.grid(True, alpha=0.3)
    
    # 6. Top 10 Configurations
    ax6 = axes[1, 2]
    top_10 = results_df.nsmallest(10, 'mse')
    config_names = [f"{row['model']}\n{'+'.join(row['features'][:2])}" for _, row in top_10.iterrows()]
    bars6 = ax6.barh(range(len(top_10)), top_10['mse'])
    ax6.set_yticks(range(len(top_10)))
    ax6.set_yticklabels(config_names, fontsize=8)
    ax6.set_xlabel('MSE')
    ax6.set_title('Top 10 Configurations')
    ax6.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('enhanced_inflation_results_original_architecture.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Print comprehensive report
    print(f"\n📋 COMPREHENSIVE RESULTS SUMMARY")
    print("="*60)
    print(f"Total experiments conducted: {len(results_df)}")
    print(f"Models tested: {results_df['model'].nunique()}")
    print(f"Feature combinations: {results_df['n_features'].value_counts().to_dict()}")
    
    print(f"\n🏆 TOP 5 CONFIGURATIONS (Lowest MSE):")
    top_5 = results_df.nsmallest(5, 'mse')
    for i, (_, row) in enumerate(top_5.iterrows(), 1):
        print(f"{i}. {row['model']} | Features: {'+'.join(row['features'])} | Scaler: {row['scaler']}")
        print(f"   MSE: {row['mse']:.6f} | RMSE: {row['rmse']:.6f} | MAE: {row['mae']:.6f}")
        print(f"   R²: {row['r2']:.6f} | MAPE: {row['mape']:.2f}%")
        print("-" * 60)
    
    print(f"\n🤖 MODEL PERFORMANCE RANKING (by average R²):")
    model_ranking = results_df.groupby('model').agg({
        'mse': 'mean',
        'r2': 'mean', 
        'mae': 'mean'
    }).round(6).sort_values('r2', ascending=False)
    print(model_ranking)
    
else:
    print("❌ No results generated!")

print(f"\n✅ ENHANCED ANALYSIS COMPLETED!")
print("🎯 All reviewer comments addressed while maintaining original architecture")